# External pressure–volume challenge
Run in Edge Colab. This challenges the native chamber pressure law using measured murine volumes, not a circulation or human clinical validation.
Source DOI: https://doi.org/10.1007/s10439-023-03293-z . Mouse 2 and 3 values remain unopened. The publisher-normalized beat split is correlated and retrospective.
The original run found worse pressure errors than the repeating training-waveform baseline. Preserve that negative result. No outer-score tuning is authorized by this recipe.


In [ ]:
from pathlib import Path
import subprocess, sys, urllib.request, os
ROOT=Path("/content/HeartTwin_external_challenge")
ROOT.mkdir(exist_ok=True)
repo=ROOT/"Virelion-HeartTwin"
if not repo.exists():
    subprocess.run(["git","clone","https://github.com/Virelion-Biotech/Virelion-HeartTwin.git",str(repo)],check=True,capture_output=True)
subprocess.run(["git","-C",str(repo),"checkout","d9e6681b9a4ec96707aed3f7884162277156f536"],check=True,capture_output=True)
env=dict(os.environ,PYTHONPATH=str(repo),OPENBLAS_NUM_THREADS="1",MPLBACKEND="Agg")
log=(ROOT/"challenge.log").open("w")
process=subprocess.run([sys.executable,str(repo/"scripts/challenge_external_pv.py"),"--output",str(ROOT/"results")],cwd=repo,env=env,stdout=log,stderr=subprocess.STDOUT)
process.check_returncode()
print("Completed external PV challenge",process.returncode,"at source 0473163bc516edaab3a89c9c7a6dea4fe1e3676b")



In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
from scipy.io import loadmat
from pathlib import Path
R=ROOT/"results"
models=json.loads((R/"frozen_models.json").read_text())
result=json.loads((R/"results.json").read_text())
data=loadmat(R/"m1_timeseries_final.mat")
pred=json.loads((R/"frozen_outer_predictions.json").read_text())
diagnosis={"purpose":"Post-score diagnosis only; no model or outer-score-selected refit.",
"source":"HeartTwin native chamber_pressure; no CardiFlow/CardiMech/CardiEP engine called",
"outer_predictions_frozen_before_pressure_scoring":True,"chambers":{}}
fig,axes=plt.subplots(2,2,figsize=(11,7),constrained_layout=True)
phase=np.linspace(0,1,50)
for row,ch in enumerate(["LV","RV"]):
    m=models[ch]; p=m["parameters"]; T=.1034
    V=data["V"+ch+"_stack"][:,40:50]/1000
    measured=data["P"+ch+"_stack"][:,40:50]
    elapsed=np.remainder(phase*T-p["activation_start_s"],T)
    duration=p["activation_duration_s"]; peak=duration*p.get("activation_peak_fraction",.5)
    a=np.where(elapsed<duration,np.sin(np.pi/2*np.where(elapsed<=peak,elapsed/peak,(duration-elapsed)/(duration-peak)))**2,0)
    delta=V-p["unstressed_volume_mL"]
    k=p.get("passive_exponent_per_mL",0.)
    passive=p["emin_mmHg_per_mL"]*delta if k==0 else p["emin_mmHg_per_mL"]*np.expm1(k*delta)/k
    reference=p["external_pressure_mmHg"]+(1-a[:,None])*passive+a[:,None]*p["emax_mmHg_per_mL"]*delta
    native=np.array(pred[ch])
    maxdiff=float(np.max(abs(reference-native)))
    assert maxdiff<1e-10
    selected=m["families"][m["selected_index"]]
    training=data["P"+ch+"_stack"][:,:40].mean(axis=1)
    diagnosis["chambers"][ch]={
      "independent_formula_max_difference_mmHg":maxdiff,
      "selected_stiffness_times_vref":selected["stiffness_times_vref"],
      "selected_asymmetric":selected["asymmetric"],
      "inner_rmse_all_families_mmHg":[f["inner_pressure_rmse_mmHg"] for f in m["families"]],
      "refit_active_mask":m["refit"]["active_mask"],
      "converged_trials":sum(t["success"] for f in m["families"] for t in f["calibration_trials"])+sum(t["success"] for t in m["refit_trials"]),
      "total_trials":sum(len(f["calibration_trials"]) for f in m["families"])+len(m["refit_trials"]),
      "pressure_range_mmHg":[float(measured.min()),float(measured.max())],
      "volume_range_mL":[float(V.min()),float(V.max())],
      "mean_error_by_phase_mmHg":(native-measured).mean(axis=1).tolist(),
      "diagnosis":"Formula agrees; remaining mismatch consistent with restricted phase/volume constitutive model and processed-beat variation. No unique physiological cause identified.",
      "baseline_comparison":"A repeating training waveform beats the fitted constitutive model on these processed beats; no predictive superiority established."}
    ax=axes[row,0]
    ax.plot(phase,measured,color=".75",alpha=.5)
    ax.plot(phase,measured.mean(axis=1),"k",label="Reserved-beat mean")
    ax.plot(phase,native.mean(axis=1),"C3",label="Native conditional pressure")
    ax.plot(phase,training,"C0",ls="--",label="Training waveform baseline")
    ax.set(xlabel="Publisher-normalized phase",ylabel=ch+" pressure (mmHg)")
    ax.legend(fontsize=8)
    ax=axes[row,1]
    ax.plot(phase,native-measured,alpha=.4,color="C3")
    ax.axhline(0,color="k",lw=.7)
    ax.set(xlabel="Publisher-normalized phase",ylabel=ch+" pressure residual (mmHg)")
fig.suptitle("External murine PV challenge: one animal, 10 reserved processed beats\nMeasured volume is an input; no circulation or human accuracy claim",fontsize=11)
fig.savefig(R/"external_pv_challenge.png",dpi=160)
(R/"diagnosis.json").write_text(json.dumps(diagnosis,indent=2,allow_nan=False))
print(json.dumps(diagnosis,indent=2))
display(fig)



In [ ]:
import subprocess, sys, json, hashlib, base64, zipfile
from pathlib import Path
subprocess.run([sys.executable,"-m","pip","install","-q","ruff"],check=True,capture_output=True)
checks={}
for name,cmd in {
"compile":[sys.executable,"-m","py_compile",str(repo/"scripts/challenge_external_pv.py")],
"lint":[sys.executable,"-m","ruff","check",str(repo/"scripts/challenge_external_pv.py")],
}.items():
    p=subprocess.run(cmd,cwd=repo,text=True,capture_output=True)
    checks[name]={"returncode":p.returncode,"stdout":p.stdout,"stderr":p.stderr}
(ROOT/"checks.json").write_text(json.dumps(checks,indent=2))
print(json.dumps(checks,indent=2))
print("CHALLENGE_EXIT",process.returncode)
print("RESERVED SUBJECT VALUES STILL UNOPENED")



## EDGAR follow-up
The signed-in 2026 challenge page lists new Bratislava training waveforms and geometry. The archive audit below is a separate source qualification, not an electrical solve. Never download challenge testing archives for model development.


In [ ]:
import urllib.request, zipfile, io, hashlib, json, time
from scipy.io import whosmat
ED=ROOT/"edgar-2026-training"
ED.mkdir(exist_ok=True)
ed_sources={
"geometries":"https://edgar.sci.utah.edu/datasets/2026_ECGI_Challenge/Training_Data/Bratislava_Data/Geometries/TrainingGeometries.zip",
"signals":"https://edgar.sci.utah.edu/datasets/2026_ECGI_Challenge/Training_Data/Bratislava_Data/Signals/AllTrainingSignals.zip"}
def audit_edgar():
    report={"page":"https://edgar.sci.utah.edu/2026/05/02/2026-ecgi-data-challenge/",
    "scope":"Bratislava training only; challenge testing NOT downloaded",
    "p034_legacy_page":"still lists Documentation and Meshes only",
    "rights":"No raw archives or recording values redistributed by this audit",
    "archives":{}}
    for kind,url in ed_sources.items():
        path=ED/(kind+".zip")
        try:
            with urllib.request.urlopen(url,timeout=120) as response, path.open("wb") as f:
                total=0
                while True:
                    chunk=response.read(1024*1024)
                    if not chunk: break
                    total+=len(chunk)
                    if total>512*1024*1024: raise ValueError("Archive exceeds declared 512MiB budget")
                    f.write(chunk)
            raw=path.read_bytes()
            with zipfile.ZipFile(io.BytesIO(raw)) as z:
                entries=[]
                for name in z.namelist():
                    item={"path":name,"bytes":z.getinfo(name).file_size}
                    if name.lower().endswith(".mat"):
                        item["mat_schema"]=[{"variable":n,"shape":list(s),"class":c} for n,s,c in whosmat(io.BytesIO(z.read(name)))]
                    entries.append(item)
            report["archives"][kind]={"url":url,"bytes":len(raw),"sha256":hashlib.sha256(raw).hexdigest(),"entries":entries}
        except Exception as exc:
            report["archives"][kind]={"url":url,"error":type(exc).__name__+": "+str(exc)}
        (ED/"training_archive_audit.json").write_text(json.dumps(report,indent=2))
    return report
from concurrent.futures import ThreadPoolExecutor
ed_pool=ThreadPoolExecutor(max_workers=1)
ed_future=ed_pool.submit(audit_edgar)
print("EDGAR training archive acquisition started; testing data excluded")
print("Training acquisition future:",ed_future.done())



In [ ]:
print("EDGAR acquisition complete:",ed_future.done())
if ed_future.done():
    print(json.dumps(ed_future.result(),indent=2))
